In [1]:
import pandas as pd
import numpy as np

In [2]:
df = pd.read_csv('../data/raw/phdb_dinosaur_occurrences.csv')
df.head()

C:\Users\john_\AppData\Local\Temp\ipykernel_19200\3316100536.py:1: DtypeWarning: Columns (56,81,82,83,84,119,131) have mixed types. Specify dtype option on import or set low_memory=False.
  df = pd.read_csv('../data/raw/phdb_dinosaur_occurrences.csv')


,occurrence_no,record_type,collection_no,identified_name,identified_rank,accepted_name,accepted_attr,accepted_rank,accepted_no,early_interval,...,max_ma_error,rare_body_parts,min_ma_error,direct_ma_value,direct_ma_unit,direct_ma_method,zone_type,direct_ma_error,plant_organ,artifacts
0,41524,occ,3257,Aves indet.,class,Aves,(Linnaeus 1758),class,36616,Lutetian,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,41580,occ,3256,Aves indet.,class,Aves,(Linnaeus 1758),class,36616,Ypresian,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2,130209,occ,10755,Chaoyangosaurus liaosiensis n. gen. n. sp.,species,Chaoyangsaurus youngi,Zhao et al. 1999,species,55580,Late Kimmeridgian,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
3,130294,occ,10764,Protarchaeopteryx robusta n. gen. n. sp.,species,Protarchaeopteryx robusta,Ji and Ji 1997,species,66068,Late Barremian,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
4,130295,occ,10764,Caudipteryx zoui n. gen. n. sp.,species,Caudipteryx zoui,Ji et al. 1998,species,66066,Late Barremian,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN


The raw PBDB dataset contains many specialized fields that are not required
for the initial analysis.

Rather than removing fields solely based on missingness, fields will be
retained or removed based on their relevance to the project's analytical
questions.

The initial analytical domains are:

- Occurrence identification
- Taxonomy
- Geological time
- Geography
- Geology
- Paleoenvironment
- Preservation
- Data provenance

In [3]:
analysis_columns = [
    # Identification
    "occurrence_no",
    "record_type",
    "collection_no",
    "identified_name",
    "identified_rank",
    "accepted_name",
    "accepted_rank",
    "accepted_no",

    # Geological time
    "early_interval",
    "late_interval",
    "max_ma",
    "min_ma",

    # Taxonomy
    "phylum",
    "class",
    "order",
    "family",
    "genus",

    # Geography
    "cc",
    "state",
    "county",
    "lat",
    "lng",
    "latlng_basis",
    "latlng_precision",
    "geogscale",
    "paleolat",
    "paleolng",
    "geoplate",

    # Geology
    "formation",
    "geological_group",
    "stratscale",
    "lithology1",
    "lithology2",
    "environment",
    "tectonic_setting",

    # Preservation
    "pres_mode",
    "preservation_quality",
    "lagerstatten",
    "collection_coverage",
    "collection_type",

    # Provenance
    "ref_author",
    "ref_pubyr",
    "reference_no",
    "research_group"
]

In [37]:
#Set column groups to break up cleaning tasks

identification_columns = [
    "occurrence_no",
    "record_type",
    "collection_no",
    "identified_name",
    "identified_rank",
    "accepted_name",
    "accepted_rank",
    "accepted_no"
]

geological_time_columns = [
    "early_interval",
    "late_interval",
    "max_ma",
    "min_ma",
]

taxonomy_columns = [
    "phylum",
    "class",
    "order",
    "family",
    "genus",
]

geography_columns = [
    "cc",
    "state",
    "county",
    "lat",
    "lng",
    "latlng_basis",
    "latlng_precision",
    "geogscale",
    "paleolat",
    "paleolng",
    "geoplate",
]

geology_columns = [
    "formation",
    "geological_group",
    "stratscale",
    "lithology1",
    "lithology2",
    "environment",
    "tectonic_setting",
]

preservation_columns = [
    "pres_mode",
    "preservation_quality",
    "lagerstatten",
    "collection_coverage",
    "collection_type",
]

provenance_columns = [
    "ref_author",
    "ref_pubyr",
    "reference_no",
    "research_group"
]

In [38]:
missing_columns = [
    col for col in analysis_columns
    if col not in df.columns
]

missing_columns

[]

In [39]:
clean_df = df[analysis_columns].copy()

In [40]:
clean_df.shape

(37835, 44)

In [41]:
#Check for N/A values in the analysis columns
clean_df.isna().sum().sort_values(ascending=False).head(20)

tectonic_setting        36676
lagerstatten            36339
collection_coverage     35351
lithology2              31453
late_interval           29152
geological_group        27457
preservation_quality    26696
county                  19068
formation               14330
genus                   14262
paleolat                11704
paleolng                11704
family                   8739
stratscale               8381
order                    7392
geogscale                6629
state                    6319
research_group           1828
latlng_basis             1612
lithology1               1177
dtype: int64

In [42]:
#Check for blanks in analysis columns
(clean_df == "").sum().sort_values(ascending=False).head(20)

occurrence_no      0
record_type        0
collection_no      0
identified_name    0
identified_rank    0
accepted_name      0
accepted_rank      0
accepted_no        0
early_interval     0
late_interval      0
max_ma             0
min_ma             0
phylum             0
class              0
order              0
family             0
genus              0
cc                 0
state              0
county             0
dtype: int64

In [43]:
#Check for known variable 'not reported' in analysis columns
(clean_df == "not reported").sum().sort_values(ascending=False).head(20)

lithology1         12047
occurrence_no          0
collection_no          0
record_type            0
identified_rank        0
accepted_name          0
accepted_rank          0
accepted_no            0
early_interval         0
late_interval          0
max_ma                 0
identified_name        0
min_ma                 0
phylum                 0
order                  0
class                  0
genus                  0
cc                     0
state                  0
family                 0
dtype: int64

In [44]:
#Check lithology1 for values and fix not reported to NaN

clean_df = clean_df.replace("not reported", np.nan)
clean_df["lithology1"].value_counts(dropna=False).head(20)

lithology1
NaN                                13224
sandstone                          11080
"siliciclastic"                     2569
mudstone                            2091
claystone                           1876
siltstone                           1775
"limestone"                          937
"shale"                              613
conglomerate                         605
marl                                 593
tar                                  592
lime mudstone                        270
breccia                              241
"carbonate"                          202
gravel                               168
wackestone                           144
peat                                 124
"mixed carbonate-siliciclastic"      110
phosphorite                          108
grainstone                            93
Name: count, dtype: int64

In [45]:
#Check data types to prep for standarization
clean_df.dtypes

occurrence_no             int64
record_type              object
collection_no             int64
identified_name          object
identified_rank          object
accepted_name            object
accepted_rank            object
accepted_no               int64
early_interval           object
late_interval            object
max_ma                  float64
min_ma                  float64
phylum                   object
class                    object
order                    object
family                   object
genus                    object
cc                       object
state                    object
county                   object
lat                     float64
lng                     float64
latlng_basis             object
latlng_precision         object
geogscale                object
paleolat                float64
paleolng                float64
geoplate                 object
formation                object
geological_group         object
stratscale               object
litholog

In [46]:
clean_df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 37835 entries, 0 to 37834
Data columns (total 44 columns):
 #   Column                Non-Null Count  Dtype  
---  ------                --------------  -----  
 0   occurrence_no         37835 non-null  int64  
 1   record_type           37835 non-null  object 
 2   collection_no         37835 non-null  int64  
 3   identified_name       37835 non-null  object 
 4   identified_rank       37835 non-null  object 
 5   accepted_name         37835 non-null  object 
 6   accepted_rank         37835 non-null  object 
 7   accepted_no           37835 non-null  int64  
 8   early_interval        37835 non-null  object 
 9   late_interval         8683 non-null   object 
 10  max_ma                37835 non-null  float64
 11  min_ma                37835 non-null  float64
 12  phylum                37835 non-null  object 
 13  class                 37835 non-null  object 
 14  order                 30443 non-null  object 
 15  family             

In [47]:
#Change id columns to string objects to not allow for any accidental math
id_columns = [
    "occurrence_no",
    "collection_no",
    "accepted_no",
    "reference_no"
]

clean_df[id_columns] = clean_df[id_columns].astype('string')
clean_df[id_columns].dtypes

occurrence_no    string[python]
collection_no    string[python]
accepted_no      string[python]
reference_no     string[python]
dtype: object

In [48]:
#Check expected numerical columns for typing
numeric_columns = [
    "max_ma",
    "min_ma",
    "lat",
    "lng",
    "paleolat",
    "paleolng",
    "ref_pubyr"
]

clean_df[numeric_columns].dtypes

max_ma       float64
min_ma       float64
lat          float64
lng          float64
paleolat     float64
paleolng     float64
ref_pubyr    float64
dtype: object

In [49]:
clean_df[numeric_columns].describe().T

,count,mean,std,min,25%,50%,75%,max
max_ma,37835.0,71.039217,65.801009,0.011700,0.129000,72.200000,121.400000,251.902000
min_ma,37835.0,64.551779,61.601400,0.000000,0.011700,66.000000,100.500000,248.600000
lat,37835.0,25.006299,29.813294,-84.333336,18.337500,37.414722,43.585079,89.039169
lng,37835.0,-20.021169,92.786884,-179.154999,-106.717598,-18.783199,35.361799,178.677002
paleolat,26131.0,24.775819,32.045348,-86.160000,21.730000,33.900000,44.745000,89.200000
paleolng,26131.0,-0.151295,68.824775,-177.600000,-64.630000,3.010000,30.630000,178.700000
ref_pubyr,37834.0,1993.540070,29.450243,1824.000000,1988.000000,2001.000000,2012.000000,2027.000000


In [50]:
clean_df['ref_pubyr'] = clean_df['ref_pubyr'].astype('Int64')
clean_df["ref_pubyr"].dtype

Int64Dtype()

Data Quality Note 
'ref_pubyr' contains at least one publication year beyond the current calander year (2027). This value will be invested during validation rather than being removed automatically. 

In [51]:
#Checking Categorical values
categorical_columns = clean_df.select_dtypes(include="object").columns.tolist()

len(categorical_columns)

33

In [52]:
categorical_columns

['record_type',
 'identified_name',
 'identified_rank',
 'accepted_name',
 'accepted_rank',
 'early_interval',
 'late_interval',
 'phylum',
 'class',
 'order',
 'family',
 'genus',
 'cc',
 'state',
 'county',
 'latlng_basis',
 'latlng_precision',
 'geogscale',
 'geoplate',
 'formation',
 'geological_group',
 'stratscale',
 'lithology1',
 'lithology2',
 'environment',
 'tectonic_setting',
 'pres_mode',
 'preservation_quality',
 'lagerstatten',
 'collection_coverage',
 'collection_type',
 'ref_author',
 'research_group']

In [53]:
#Check and clean white space issues
whitespace_issues = {}

for col in categorical_columns:
    mask = clean_df[col].astype("string").str.strip() != clean_df[col].astype("string")
    count = mask.sum()

    if count > 0:
        whitespace_issues[col] = count

whitespace_issues

{'state': np.int64(11),
 'county': np.int64(4),
 'formation': np.int64(5),
 'geological_group': np.int64(3),
 'ref_author': np.int64(9)}

In [54]:
for col in whitespace_issues:
    print(f"\n--- {col} ---")
    
    mask = (
        clean_df[col].astype("string").str.strip()
        != clean_df[col].astype("string")
    )
    
    print(clean_df.loc[mask, col].unique())


--- state ---
[" Provence-Alpes-Côte d'Azur" 'Tarija ' ' Gyeongsangnam-do'
 'Gyeongsangnam-do ' 'central Macedonia ' 'Nouvelle-Aquitaine ' ' Neuquén'
 'Guizhou ' 'Jiangxi ' 'Chongqing ']

--- county ---
['Rockingham ' 'Baranya ' 'Huichang ' 'Middlesex ']

--- formation ---
['Mesilla Valley ' 'Itanoura ' 'Menilite ' 'Hudspeth ']

--- geological_group ---
['El Foyel ' 'Carmanah ']

--- ref_author ---
['Desjardins ']


In [55]:
clean_df[categorical_columns] = clean_df[categorical_columns].apply(
    lambda col: col.str.strip()
)

fixed_whitespace_issues = {}

for col in categorical_columns:
    mask = (
        clean_df[col].astype("string").str.strip()
        != clean_df[col].astype("string")
    )
    
    count = mask.sum()
    
    if count > 0:
        whitespace_issues[col] = count

fixed_whitespace_issues

{}

In [56]:
category_counts = (
    clean_df[categorical_columns]
    .nunique(dropna=True)
    .sort_values()
)

category_counts

record_type                 1
phylum                      1
lagerstatten                2
class                       4
geogscale                   5
stratscale                  5
latlng_basis                5
preservation_quality        6
collection_type             6
tectonic_setting           11
latlng_precision           11
accepted_rank              15
identified_rank            15
research_group             27
collection_coverage        28
lithology2                 30
lithology1                 41
order                      68
environment                70
geoplate                   72
late_interval             154
pres_mode                 160
cc                        168
early_interval            253
geological_group          314
family                    409
state                     876
county                   1307
formation                1776
genus                    3087
ref_author               3818
accepted_name            6172
identified_name         10509
dtype: int

In [57]:
# Manually check smaller category counts for any erronous values
for col in category_counts[category_counts <= 25].index:
    print(f"\n--- {col} ---")
    print(clean_df[col].value_counts(dropna=False))


--- record_type ---
record_type
occ    37835
Name: count, dtype: int64

--- phylum ---
phylum
Chordata    37835
Name: count, dtype: int64

--- lagerstatten ---
lagerstatten
NaN             36339
concentrate      1349
conservation      147
Name: count, dtype: int64

--- class ---
class
Aves            15177
Reptilia        11179
Ornithischia     7078
Saurischia       4401
Name: count, dtype: int64

--- geogscale ---
geogscale
small collection    14528
outcrop             14412
NaN                  6629
local area           2074
hand sample            98
basin                  94
Name: count, dtype: int64

--- stratscale ---
stratscale
bed              20423
NaN               8381
group of beds     7148
formation         1115
member             704
group               64
Name: count, dtype: int64

--- latlng_basis ---
latlng_basis
based on nearby landmark    12872
estimated from map          11998
stated in text               9003
NaN                          1612
based on political uni

##### No capitalization inconsistences found
##### No obvious spelling inconsistences found
##### Scientific categories preserved. 

##### latlng_precision needs further investigation to determine full meaning.
##### lagerstatten may be converted to a True/False category for ease.

In [58]:
clean_df.duplicated().sum()

np.int64(0)

In [59]:
clean_df["occurrence_no"].duplicated().sum()

np.int64(0)

In [63]:
# Validate taxonomy
clean_df[taxonomy_columns].head()

,phylum,class,order,family,genus
0,Chordata,Aves,NaN,NaN,NaN
1,Chordata,Aves,NaN,NaN,NaN
2,Chordata,Ornithischia,NO_ORDER_SPECIFIED,Chaoyangsauridae,Chaoyangsaurus
3,Chordata,Reptilia,NO_ORDER_SPECIFIED,NO_FAMILY_SPECIFIED,Protarchaeopteryx
4,Chordata,Reptilia,NO_ORDER_SPECIFIED,NO_FAMILY_SPECIFIED,Caudipteryx


In [ ]:
clean_df["phylum"].value_counts(dropna=False)

phylum
Chordata    37835
Name: count, dtype: int64

In [30]:
clean_df["class"].value_counts(dropna=False)

class
Aves            15177
Reptilia        11179
Ornithischia     7078
Saurischia       4401
Name: count, dtype: int64

In [34]:
clean_df["order"].value_counts(dropna=False)

order
NO_ORDER_SPECIFIED    14853
NaN                    7392
Passeriformes          2352
Charadriiformes        1949
Anseriformes           1821
                      ...  
Yandangithiformes         1
Mesitornithiformes        1
Praeornithiformes         1
Alcediniformes            1
Pancharadriiformes        1
Name: count, Length: 69, dtype: int64

In [31]:
clean_df["family"].value_counts(dropna=False)

family
NaN                    8739
NO_FAMILY_SPECIFIED    3047
Anatidae               1731
Hadrosauridae          1633
Dromaeosauridae        1017
                       ... 
Anachronornithidae        1
Morsoravidae              1
Diomedeoididae            1
Waltonortygidae           1
Argentinosauridae         1
Name: count, Length: 410, dtype: int64

In [35]:
clean_df['genus'].value_counts(dropna=False)

genus
NaN                14262
Grallator            431
Anas                 336
Eubrontes            304
Phalacrocorax        278
                   ...  
Mesetasaurus           1
Changzhousaurus        1
Ceratocaudia           1
Jian                   1
Plesiolophus           1
Name: count, Length: 3088, dtype: int64

In [65]:
clean_df[taxonomy_columns].isna().sum()

phylum        0
class         0
order      7392
family     8739
genus     14262
dtype: int64

In [66]:
taxonomy_hierarchy_check = {
    "genus_without_family": (
        clean_df["genus"].notna() &
        clean_df["family"].isna()
    ).sum(),

    "family_without_order": (
        clean_df["family"].notna() &
        clean_df["order"].isna()
    ).sum(),

    "order_without_class": (
        clean_df["order"].notna() &
        clean_df["class"].isna()
    ).sum(),

    "class_without_phylum": (
        clean_df["class"].notna() &
        clean_df["phylum"].isna()
    ).sum()
}

taxonomy_hierarchy_check

{'genus_without_family': np.int64(0),
 'family_without_order': np.int64(178),
 'order_without_class': np.int64(0),
 'class_without_phylum': np.int64(0)}

In [68]:
clean_df.loc[
    clean_df["family"].notna() & clean_df["order"].isna(),
    ["accepted_name", "accepted_rank", "class", "order", "family", "genus"]
].tail(20)

,accepted_name,accepted_rank,class,order,family,genus
34615,Mamenchisauridae,unranked clade,Saurischia,NaN,Mamenchisauridae,NaN
34768,Brachyrostra,unranked clade,Reptilia,NaN,Abelisauridae,NaN
34799,Mamenchisauridae,unranked clade,Saurischia,NaN,Mamenchisauridae,NaN
34841,Rebbachisauridae,unranked clade,Saurischia,NaN,Rebbachisauridae,NaN
35515,Diplodocidae,unranked clade,Saurischia,NaN,Diplodocidae,NaN
35574,Diplodocidae,unranked clade,Saurischia,NaN,Diplodocidae,NaN
35623,Saltasauridae,unranked clade,Saurischia,NaN,Saltasauridae,NaN
35737,Euhelopodidae,unranked clade,Saurischia,NaN,Euhelopodidae,NaN
35781,Rebbachisauridae,unranked clade,Saurischia,NaN,Rebbachisauridae,NaN
35861,Rebbachisauridae,unranked clade,Saurischia,NaN,Rebbachisauridae,NaN


In [69]:
rank_consistency = (
    clean_df.groupby("accepted_name")["accepted_rank"]
    .nunique()
    .sort_values(ascending=False)
)

rank_consistency.head(20)

accepted_name
Hadrosauridae                2
Megalosauridae               2
Accipiter quartus            1
Zosteropidae                 1
Zoothera dauma major         1
Abrosaurus dongpoensis       1
Zygodactylus luberonensis    1
Aardonyx celestae            1
Abavornis                    1
Abavornis bonaparti          1
Abdarainurus barsboldi       1
Abditosaurus kuehnei         1
Abelichnus astigarrae        1
Abitusavis lii               1
Abydosaurus mcintoshi        1
Acanthis                     1
Acanthisitta chloris         1
Acantholipan gonzalezi       1
Accipiter                    1
Accipiter cooperii           1
Name: accepted_rank, dtype: int64

In [74]:
clean_df.loc[
    clean_df["accepted_name"].isin(["Hadrosauridae", "Megalosauridae"]),
    ["accepted_name", "accepted_rank", "class", "order", "family", "genus"]
].drop_duplicates()

,accepted_name,accepted_rank,class,order,family,genus
8,Hadrosauridae,family,Ornithischia,NO_ORDER_SPECIFIED,Hadrosauridae,NaN
2792,Hadrosauridae,unranked clade,Ornithischia,NaN,Hadrosauridae,NaN
4258,Megalosauridae,unranked clade,Reptilia,NaN,Megalosauridae,NaN
8087,Megalosauridae,family,Reptilia,NO_ORDER_SPECIFIED,Megalosauridae,NaN


In [75]:
# Validate non-dinosaurian classes
clean_df.loc[
    clean_df["class"].isin(["Aves", "Reptilia"]),
    ["accepted_name", "accepted_rank", "class", "order", "family", "genus"]
].head(30)

,accepted_name,accepted_rank,class,order,family,genus
0,Aves,class,Aves,NaN,NaN,NaN
1,Aves,class,Aves,NaN,NaN,NaN
3,Protarchaeopteryx robusta,species,Reptilia,NO_ORDER_SPECIFIED,NO_FAMILY_SPECIFIED,Protarchaeopteryx
4,Caudipteryx zoui,species,Reptilia,NO_ORDER_SPECIFIED,NO_FAMILY_SPECIFIED,Caudipteryx
5,Theropoda,unranked clade,Reptilia,NaN,NaN,NaN
6,Dinosauria,unranked clade,Reptilia,NaN,NaN,NaN
7,Gorgosaurus libratus,species,Reptilia,NO_ORDER_SPECIFIED,Tyrannosauridae,Gorgosaurus
9,Gorgosaurus libratus,species,Reptilia,NO_ORDER_SPECIFIED,Tyrannosauridae,Gorgosaurus
11,Gorgosaurus libratus,species,Reptilia,NO_ORDER_SPECIFIED,Tyrannosauridae,Gorgosaurus
12,Gorgosaurus libratus,species,Reptilia,NO_ORDER_SPECIFIED,Tyrannosauridae,Gorgosaurus


In [76]:
clean_df["accepted_name"].str.contains(
    "Dinosauria",
    case=False,
    na=False
).sum()

np.int64(1422)

In [77]:
clean_df["accepted_name"].value_counts().head(20)

accepted_name
Theropoda          1995
Dinosauria         1422
Sauropoda          1041
Hadrosauridae      1031
Aves                633
Ornithopoda         521
Ceratopsidae        291
Dromaeosauridae     264
Titanosauria        249
Tyrannosauridae     245
Ankylosauria        240
Ornithischia        199
Grallator           195
Passeriformes       167
Coelurosauria       164
Ornithomimidae      159
Titanosauridae      155
Ankylosauridae      149
Ceratopsia          148
Abelisauridae       140
Name: count, dtype: int64

Taxonomic clarification: PBDB class field should not be used as a dinosaur/non-dinsaur filter because dinosaur occurences may be classified under Reptilia, while AVes may be represeted separtetly. Dinosaur membership will be defined using PBDB taxonomy rather than class alone. 

No taxonmy records needed modified. All relevant data is needed and any missing or unknown data is done purposefully. 

In [78]:
# Validate geological time

age_check = {
    "max_less_than_min": (
        clean_df["max_ma"] < clean_df["min_ma"]
    ).sum(),

    "max_missing": clean_df["max_ma"].isna().sum(),

    "min_missing": clean_df["min_ma"].isna().sum(),

    "both_missing": (
        clean_df["max_ma"].isna() &
        clean_df["min_ma"].isna()
    ).sum()
}

age_check

{'max_less_than_min': np.int64(0),
 'max_missing': np.int64(0),
 'min_missing': np.int64(0),
 'both_missing': np.int64(0)}

In [79]:
clean_df.loc[
    clean_df["max_ma"].nlargest(10).index,
    ["accepted_name", "early_interval", "late_interval", "max_ma", "min_ma"]
]

,accepted_name,early_interval,late_interval,max_ma,min_ma
31596,Deuterotetrapous triassicus,Early Triassic,NaN,251.902,247.0
21134,Coelurosaurichnus ziegelangernensis,Olenekian,NaN,250.800,247.0
21135,Coelurosaurichnus,Olenekian,NaN,250.800,247.0
21136,Coelurosaurichnus,Olenekian,NaN,250.800,247.0
25377,Grallator,Smithian,NaN,250.800,248.6
25378,Eubrontes,Smithian,NaN,250.800,248.6
612,Coelurosaurichnus perriauxi,Anisian,Ladinian,247.000,237.0
613,Anchisauripus bibractensis,Anisian,Ladinian,247.000,237.0
937,Coelurosaurichnus sabinensis,Middle Triassic,NaN,247.000,237.0
991,Coelurosaurichnus perriauxi,Anisian,Ladinian,247.000,237.0


In [80]:
clean_df.loc[
    clean_df["min_ma"].nsmallest(10).index,
    ["accepted_name", "early_interval", "late_interval", "max_ma", "min_ma"]
]

,accepted_name,early_interval,late_interval,max_ma,min_ma
130,Aves,Holocene,NaN,0.0117,0.0
144,Aves,Pleistocene,Holocene,2.5800,0.0
145,Struthio camelus,Holocene,NaN,0.0117,0.0
146,Struthio camelus,Holocene,NaN,0.0117,0.0
147,Columbidae,Holocene,NaN,0.0117,0.0
148,Struthio camelus,Holocene,NaN,0.0117,0.0
177,Struthio camelus,Holocene,NaN,0.0117,0.0
178,Streptopelia,Holocene,NaN,0.0117,0.0
179,Struthio camelus,Holocene,NaN,0.0117,0.0
180,Struthio camelus,Holocene,NaN,0.0117,0.0


In [82]:
clean_df.groupby(
    ["early_interval", "late_interval"],
    dropna=False
)[["max_ma", "min_ma"]].agg(["min", "max"]).head(30)

max_ma           min_ma         
                                    min     max      min      max
early_interval late_interval                                     
Aalenian       Bajocian          174.70  174.70  168.200  168.200
               Early Bajocian    174.70  174.70  168.600  168.600
               NaN               174.70  174.70  170.900  170.900
Alaunian       Sevatian          215.38  215.38  205.700  205.700
               NaN               215.38  215.38  211.180  211.180
Albian         Campanian         113.20  113.20   72.200   72.200
               Cenomanian        113.20  113.20   93.900   93.900
               Coniacian         113.20  113.20   85.700   85.700
               Early Cenomanian  113.20  113.20   93.900   93.900
               Turonian          113.20  113.20   89.800   89.800
               NaN               113.20  113.20  100.500  100.500
Altonian       NaN                18.70   18.70   15.900   15.900
Anisian        Carnian           247.00  247.00  227.300  227.300
               Ladinian          247.00  247.00  237.000  237.000
               NaN               247.00  247.00  241.464  241.464
Aptian         Albian            121.40  121.40  100.500  100.500
               Campanian         121.40  121.40   72.200   72.200
               Cenomanian        121.40  121.40   93.900   93.900
               Early Albian      121.40  121.40  110.100  110.100
               Early Cenomanian  121.40  121.40   93.900   93.900
               Late Albian       121.40  121.40  100.500  100.500
               NaN               121.40  121.40  113.200  113.200
Aquitanian     Burdigalian        23.04   23.04   15.980   15.980
               Langhian           23.04   23.04   13.820   13.820
               Miocene            23.04   23.04    5.333    5.333
               NaN                23.04   23.04   20.450   20.450
Arikareean     Hemingfordian      29.50   29.50   16.300   16.300
               NaN                29.50   29.50   18.500   18.500
Badenian       NaN                13.82   13.82   12.800   12.800
Bajocian       Bathonian         170.90  170.90  165.300  165.300

In [83]:
clean_df[["max_ma", "min_ma"]].stack().round(6).value_counts().head(20)

0.0117      10090
72.2000      8395
0.0000       6220
83.6000      5277
0.1290       5252
66.0000      5054
100.5000     2448
143.1000     2176
121.4000     2033
93.9000      1473
113.2000     1436
125.7700     1334
154.8000     1301
201.4000     1297
149.2000     1181
0.7740        985
2.5800        954
192.9000      898
137.0500      708
132.6000      689
Name: count, dtype: int64

In [84]:
clean_df["age_range_ma"] = clean_df["max_ma"] - clean_df["min_ma"]

clean_df["age_range_ma"].describe()

count    37835.000000
mean         6.487438
std          7.700408
min          0.011700
25%          0.117300
50%          6.050000
75%         11.200000
max        167.280000
Name: age_range_ma, dtype: float64

In [85]:
clean_df["age_range_ma"].nlargest(20)

747      167.28
31817    135.40
34716    135.40
35001    135.40
8720     100.90
9156      93.90
42        77.10
1704      77.10
6864      77.10
6865      77.10
7097      77.10
7347      77.10
7686      77.10
9597      77.10
9599      77.10
9974      77.10
11793     77.10
14774     77.10
18342     77.10
18343     77.10
Name: age_range_ma, dtype: float64

In [86]:
clean_df.nlargest(20, "age_range_ma")[
    ["accepted_name", "early_interval", "late_interval", "max_ma", "min_ma", "age_range_ma"]
]

,accepted_name,early_interval,late_interval,max_ma,min_ma,age_range_ma
747,Lewisuchus admixtus,Longobardian,Judithian,239.48,72.2,167.28
31817,Dinosauria,Jurassic,Cretaceous,201.40,66.0,135.40
34716,Ornithopoda,Jurassic,Cretaceous,201.40,66.0,135.40
35001,Sauropoda,Jurassic,Cretaceous,201.40,66.0,135.40
8720,Dinosauria,Jurassic,Early Cretaceous,201.40,100.5,100.90
9156,Dinosauria,Late Triassic,Jurassic,237.00,143.1,93.90
42,Dinosauria,Cretaceous,NaN,143.10,66.0,77.10
1704,Faveoloolithidae,Cretaceous,NaN,143.10,66.0,77.10
6864,Dinosauria,Cretaceous,NaN,143.10,66.0,77.10
6865,Sauropoda,Cretaceous,NaN,143.10,66.0,77.10


In [87]:
clean_df.drop(columns="age_range_ma", inplace=True)

No missing numeric age bounds, no backwards max/min ranges, no obviously wrong ages found. Geological intervals are labeled consistently. Removed age_range_ma due to it being a diagnostic column. 

In [88]:
# Validate geography

clean_df[geography_columns].isna().sum()

cc                     17
state                6319
county              19068
lat                     0
lng                     0
latlng_basis         1612
latlng_precision        0
geogscale            6629
paleolat            11704
paleolng            11704
geoplate                0
dtype: int64